# Home & Kitchen product metadata sample

Stream the compressed Amazon metadata, retain only the product ID, description, and primary image URLs, and build a reproducible 10,000-product sample without loading the full dataset into memory.

In [13]:
from pathlib import Path
import csv
import gzip
import json
import random

from IPython.display import Image, display
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SOURCE_PATH = PROJECT_ROOT / "full_metadata" / "meta_Home_and_Kitchen.jsonl.gz"
SAMPLE_PATH = PROJECT_ROOT / "data" / "meta_Home_and_Kitchen_sample_10k.csv"
SAMPLE_SIZE = 10_000
RANDOM_SEED = 42

SOURCE_PATH, SAMPLE_PATH

(PosixPath('/Users/vachemacbook/Desktop/RecSystem/RecSystem/full_metadata/meta_Home_and_Kitchen.jsonl.gz'),
 PosixPath('/Users/vachemacbook/Desktop/RecSystem/RecSystem/data/meta_Home_and_Kitchen_sample_10k.csv'))

In [ ]:
def normalize_product(record):
    descriptions = record.get("description") or []
    if isinstance(descriptions, str):
        descriptions = [descriptions]
    description = " ".join(part.strip() for part in descriptions if isinstance(part, str) and part.strip())

    images = record.get("images") or []
    primary = next((image for image in images if image.get("variant") == "MAIN"), images[0] if images else {})
    image_url = primary.get("large") or primary.get("hi_res") or primary.get("thumb")
    image_url_high_res = primary.get("hi_res") or primary.get("large") or primary.get("thumb")

    product = {
        "asin": record.get("parent_asin") or record.get("asin"),
        "description": description,
        "image_url": image_url,
        "image_url_high_res": image_url_high_res,
    }
    return product if product["asin"] and description and image_url else None


def reservoir_sample(path, sample_size=10_000, seed=42):
    rng = random.Random(seed)
    sample = []
    eligible_count = 0

    with gzip.open(path, "rt", encoding="utf-8") as source:
        for line_number, line in enumerate(source, start=1):
            product = normalize_product(json.loads(line))
            if product is None:
                continue

            eligible_count += 1
            if len(sample) < sample_size:
                sample.append(product)
            else:
                replacement_index = rng.randrange(eligible_count)
                if replacement_index < sample_size:
                    sample[replacement_index] = product

            if line_number % 500_000 == 0:
                print(f"Read {line_number:,} products; {eligible_count:,} eligible")

    return sample, eligible_count

In [14]:
if SAMPLE_PATH.exists():
    print(f"Using existing sample: {SAMPLE_PATH}")
else:
    products, eligible_count = reservoir_sample(SOURCE_PATH, SAMPLE_SIZE, RANDOM_SEED)
    SAMPLE_PATH.parent.mkdir(parents=True, exist_ok=True)
    with SAMPLE_PATH.open("w", newline="", encoding="utf-8") as output:
        writer = csv.DictWriter(output, fieldnames=["asin", "description", "image_url", "image_url_high_res"])
        writer.writeheader()
        writer.writerows(products)
    print(f"Saved {len(products):,} sampled products from {eligible_count:,} eligible products to {SAMPLE_PATH}")

Using existing sample: /Users/vachemacbook/Desktop/RecSystem/RecSystem/data/meta_Home_and_Kitchen_sample_10k.csv


In [15]:
sample_df = pd.read_csv(SAMPLE_PATH)
print(f"Rows: {len(sample_df):,}")
display(sample_df.head(10))

for row in sample_df.head(5).itertuples():
    print(f"{row.asin}: {row.description[:250]}")
    display(Image(url=row.image_url, width=250))

Rows: 10,000


,asin,description,image_url,image_url_high_res
0,B00AT1G32Y,"Enjoy a cool and comfortable night's sleep with exceptionally thin breathable sheets that have a silky soft feel and a lustrous finish. These sheets provide a lasting vibrancy of color no matter how often thay are washed, and are made of fine yar...",https://m.media-amazon.com/images/I/41v0Du4gBaL._AC_.jpg,https://m.media-amazon.com/images/I/71d4fXA1-mL._AC_SL1500_.jpg
1,B09JKDNRRF,"We provide you with unique designs and high-quality household products at affordable prices.We are committed to making your home more warm and lovely, making your life easier and more comfortable.When designing each product, we not only consider ...",https://m.media-amazon.com/images/I/61vObrIEIIL._AC_.jpg,https://m.media-amazon.com/images/I/814pvuPvy5L._AC_SL1500_.jpg
2,B01KP46PWW,"Full Tang Cleaver. Triple riveted handle. 11 3/4"" total length. Reinforced plastic grip. 14 ounces.",https://m.media-amazon.com/images/I/31ykqxzdntL._AC_.jpg,https://m.media-amazon.com/images/I/71cb7ExJxFL._AC_SL1500_.jpg
3,B007KQ6GHI,"Henckels International makes essential kitchen tools every home chef needs. With HI Flatware, this trusted brand brings over 120 years of cutlery expertise to the world of flatware. Each piece is fabricated from high-quality 18/10 stainless steel...",https://m.media-amazon.com/images/I/318x5ExJ9uS._AC_.jpg,https://m.media-amazon.com/images/I/61K0ijDe1AS._AC_SL1500_.jpg
4,B07G3684B3,Potato Ricer/Potato Masher/Stainless Steel Potato Masher/Best Potato Ricer Do you love mashed potato? But are you frustrated at how much time and how tiring it is to use a conventional potato masher? Are your mashed potatoes still have lumps and...,https://m.media-amazon.com/images/I/41a1pqURY+L._AC_.jpg,https://m.media-amazon.com/images/I/41a1pqURY+L._AC_.jpg
5,B00KD70NXY,"The padded upper back of the Orson Collection is supported by wooden slats that give way to the substantially sized seat of this whimsical accent chair. The collection has six available fabric options - Blue tonal stripe, Black & White swirl, Mul...",https://m.media-amazon.com/images/I/51X3LVtzpJL._AC_.jpg,https://m.media-amazon.com/images/I/91W9533PAbL._AC_SL1500_.jpg
6,B07PZS8713,"Made of high quality ceramic with our unique design on both sides. Everyone will love your new favorite mug while you enjoy drinking your coffee, tea or hot chocolate. A perfect gift for coffee for tea drinkers : hilarious, fun, happy and useful ...",https://m.media-amazon.com/images/I/41m1k2iz7FL._AC_.jpg,https://m.media-amazon.com/images/I/51-XQwyXLjL._AC_SL1000_.jpg
7,B08J29H3B8,Elevate your home decor and add some holiday cheer to your space with Desktop Mini Pine Tree，it's very nice with the right amount of pops of color.,https://m.media-amazon.com/images/I/41QbcjXk4uL._AC_.jpg,https://m.media-amazon.com/images/I/71XQzqLNAeL._AC_SL1500_.jpg
8,B00JZFG8SO,"Vintage by Stephanie Ryan is a beautiful and encouraging collection of gifts that come straight from the heart. With a soft calming palette, bold blossoming flowers, each piece is designed to remind you of what makes life worth living.",https://m.media-amazon.com/images/I/41ti0lNTF6L._AC_.jpg,https://m.media-amazon.com/images/I/61YujSzvMfL._AC_SL1000_.jpg
9,B097B64GZ5,Vibrant colorful rainbow floral bloom prints scattered around the chintz quilt offer that pop of color your room needs. Rather you want to refresh your bedroom or a guest room you will be pleased. Offering breath taking floral designs in various ...,https://m.media-amazon.com/images/I/61UHth7bMQL._AC_.jpg,https://m.media-amazon.com/images/I/91JCIz1fmOL._AC_SL1500_.jpg


B00AT1G32Y: Enjoy a cool and comfortable night's sleep with exceptionally thin breathable sheets that have a silky soft feel and a lustrous finish. These sheets provide a lasting vibrancy of color no matter how often thay are washed, and are made of fine yarns t


B09JKDNRRF: We provide you with unique designs and high-quality household products at affordable prices.We are committed to making your home more warm and lovely, making your life easier and more comfortable.When designing each product, we not only consider beau


B01KP46PWW: Full Tang Cleaver. Triple riveted handle. 11 3/4" total length. Reinforced plastic grip. 14 ounces.


B007KQ6GHI: Henckels International makes essential kitchen tools every home chef needs. With HI Flatware, this trusted brand brings over 120 years of cutlery expertise to the world of flatware. Each piece is fabricated from high-quality 18/10 stainless steel for


B07G3684B3: Potato Ricer/Potato Masher/Stainless Steel Potato Masher/Best Potato Ricer Do you love mashed potato? But are you frustrated at how much time and how tiring it is to use a conventional potato masher?  Are your mashed potatoes still have lumps and pie


## Download the sampled product images

Download one primary image for every sampled ASIN. Downloads are resumable: existing valid files are skipped, failures are recorded, and rerunning the cell retries only missing images.

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from urllib.request import Request, urlopen
from urllib.error import HTTPError, URLError
import time

IMAGE_DIR = PROJECT_ROOT / "data" / "images" / "home_and_kitchen_10k"
IMAGE_MANIFEST_PATH = PROJECT_ROOT / "data" / "meta_Home_and_Kitchen_images_10k.csv"
IMAGE_FAILURES_PATH = PROJECT_ROOT / "data" / "meta_Home_and_Kitchen_image_failures_10k.csv"
MAX_WORKERS = 24
IMAGE_DIR.mkdir(parents=True, exist_ok=True)

def image_extension(content):
    if content.startswith(b"\xff\xd8\xff"):
        return ".jpg"
    if content.startswith(b"\x89PNG\r\n\x1a\n"):
        return ".png"
    if content[:4] == b"RIFF" and content[8:12] == b"WEBP":
        return ".webp"
    if content.startswith((b"GIF87a", b"GIF89a")):
        return ".gif"
    raise ValueError("Response is not a supported image")

def download_product_image(row, attempts=3):
    asin = row.asin
    existing = next((path for path in IMAGE_DIR.glob(f"{asin}.*") if path.stat().st_size > 0), None)
    if existing:
        return {"asin": asin, "image_url": row.image_url, "image_path": str(existing.relative_to(PROJECT_ROOT)), "error": ""}

    for attempt in range(1, attempts + 1):
        try:
            request = Request(row.image_url, headers={"User-Agent": "Mozilla/5.0"})
            with urlopen(request, timeout=30) as response:
                content = response.read()
            suffix = image_extension(content)
            destination = IMAGE_DIR / f"{asin}{suffix}"
            temporary = destination.with_suffix(destination.suffix + ".part")
            temporary.write_bytes(content)
            temporary.replace(destination)
            return {"asin": asin, "image_url": row.image_url, "image_path": str(destination.relative_to(PROJECT_ROOT)), "error": ""}
        except (HTTPError, URLError, TimeoutError, ValueError, OSError) as exc:
            if attempt == attempts:
                return {"asin": asin, "image_url": row.image_url, "image_path": "", "error": str(exc)}
            time.sleep(attempt)

records = list(sample_df[["asin", "image_url"]].itertuples(index=False))
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    download_results = list(executor.map(download_product_image, records))

results_df = pd.DataFrame(download_results)
image_manifest_df = results_df.loc[results_df.error.eq(""), ["asin", "image_url", "image_path"]]
failures_df = results_df.loc[results_df.error.ne(""), ["asin", "image_url", "error"]]
image_manifest_df.to_csv(IMAGE_MANIFEST_PATH, index=False)
failures_df.to_csv(IMAGE_FAILURES_PATH, index=False)
print(f"Downloaded: {len(image_manifest_df):,} / {len(records):,}")
print(f"Failures: {len(failures_df):,}")

In [ ]:
display(image_manifest_df.head(10))

# SigLIP2-ready pairs: descriptions stay in the sample table and join to local images by ASIN.
siglip2_df = sample_df[["asin", "description"]].merge(
    image_manifest_df[["asin", "image_path"]],
    on="asin",
    how="inner",
)
SIGLIP2_PAIRS_PATH = PROJECT_ROOT / "data" / "meta_Home_and_Kitchen_siglip2_pairs_10k.csv"
siglip2_df.to_csv(SIGLIP2_PAIRS_PATH, index=False)
print(f"Usable image-text pairs: {len(siglip2_df):,}")
display(siglip2_df.head())

for row in siglip2_df.head(5).itertuples():
    print(f"{row.asin}: {row.description[:250]}")
    display(Image(filename=str(PROJECT_ROOT / row.image_path), width=250))